# Controlled gates: CNOT, CZ, CY

Controlled gates act on two qubits.  The **control** qubit decides
whether the gate fires on the **target** qubit.

- **CNOT (CX)**: flips target if control is $|1\rangle$.
- **CZ**: adds $-1$ phase if both qubits are $|1\rangle$.
- **CY**: applies $Y$ to target if control is $|1\rangle$.

In [ ]:
from IPython.display import display
import qiskit as qk
import qiskit_aer as qka

In [ ]:
def show(qc, title):
    print(title)
    print(qc.draw())
    sv = qk.quantum_info.Statevector.from_instruction(
        qc.remove_final_measurements(inplace=False)
    )
    for bits, amp in sv.to_dict().items():
        if abs(amp) > 1e-12:
            print(f"  {amp.real:+.3f}{amp.imag:+.3f}j |{bits}>")
    print()


def show_mpl(qc, title):
    print(title)
    display(qc.draw(output="mpl"))
    sv = qk.quantum_info.Statevector.from_instruction(
        qc.remove_final_measurements(inplace=False)
    )
    for bits, amp in sv.to_dict().items():
        if abs(amp) > 1e-12:
            print(f"  {amp.real:+.3f}{amp.imag:+.3f}j |{bits}>")
    print()


def truth_table(gate_name, gate_fn, num_qubits=2):
    """Print the full truth table for a gate."""
    print(f"=== {gate_name} truth table ===")
    print(f"  {'input':>8}  ->  {'output':>8}")
    print(f"  {'--------':>8}     {'--------':>8}")
    for i in range(2 ** num_qubits):
        bits = format(i, f"0{num_qubits}b")
        qc = qk.QuantumCircuit(num_qubits)
        for q in range(num_qubits):
            if bits[num_qubits - 1 - q] == "1":
                qc.x(q)
        gate_fn(qc)
        sv = qk.quantum_info.Statevector.from_instruction(qc)
        out = next(iter(sv.to_dict()))
        print(f"  |{bits}>  ->  |{out}>")
    print()

## CNOT (CX) truth table

Control = qubit 1 (left), target = qubit 0 (right).
The target flips only when the control is $|1\rangle$.

In [ ]:
truth_table("CNOT (CX)", lambda qc: qc.cx(1, 0))

## CZ truth table

$CZ$ is diagonal: it only adds a $-1$ phase to $|11\rangle$.
All other basis states pass through unchanged.

In [ ]:
print("=== CZ truth table ===")
print("  CZ is diagonal: only |11> gets a -1 phase.")
print("  |00> -> |00>,  |01> -> |01>,  |10> -> |10>,  |11> -> -|11>")
print()
for i in range(4):
    bits = format(i, "02b")
    qc = qk.QuantumCircuit(2)
    if bits[1] == "1":
        qc.x(1)
    if bits[0] == "1":
        qc.x(0)
    qc.cz(1, 0)
    sv = qk.quantum_info.Statevector.from_instruction(qc)
    for b, a in sv.to_dict().items():
        if abs(a) > 1e-10:
            print(f"  |{bits}> -> {a.real:+.1f} |{b}>")
print()

## CY truth table

$CY$ applies $Y$ to the target when the control is $|1\rangle$.
$Y|0\rangle = i|1\rangle$ and $Y|1\rangle = -i|0\rangle$.

In [ ]:
truth_table("CY", lambda qc: qc.cy(1, 0))

## CNOT with superposition input

Putting the control in superposition ($H$) and applying CNOT creates
a **Bell state** — the qubits become entangled.

In [ ]:
qc = qk.QuantumCircuit(2, 2)
qc.h(1)
qc.cx(1, 0)
show_mpl(qc, "H on control + CNOT = Bell state")

qc_m = qc.copy()
qc_m.measure([1, 0], [1, 0])
backend = qka.AerSimulator()
compiled = qk.transpile(qc_m, backend)
counts = backend.run(compiled, shots=2000).result().get_counts()
print("counts:", counts)
display(qk.visualization.plot_histogram(counts, title="Bell state measurement"))

## Summary

- **CNOT** is the workhorse of entanglement: combine $H$ + CX.
- **CZ** is symmetric (control and target are interchangeable).
- **CY** is less common but follows the same pattern.
- In Qiskit bitstrings: qubit 1 = left, qubit 0 = right.